### Import the Data

In [ ]:
%run  Data_preparation_TCGA.ipynb

### Import the Model

In [ ]:
from Classification_model import *

### Training Process

In [ ]:
train_loader = DataLoader(training_set, batch_size=1024, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=5096, shuffle=False)

In [ ]:
torch.manual_seed(0)

num_hiddens_genotype = 16
num_hiddens_final = 16

model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx))

In [ ]:
def create_term_mask(term_direct_gene_map, gene_dim, device):

    term_mask_map = {}

    for term, gene_set in term_direct_gene_map.items():

        mask = torch.zeros(len(gene_set), gene_dim)

        for i, gene_id in enumerate(gene_set):
            mask[i, gene_id] = 1

        mask_gpu = torch.autograd.Variable(mask)

        term_mask_map[term] = mask_gpu.to(device)

    return term_mask_map

term_mask_map = create_term_mask(model.term_direct_gene_map, num_genes, device = DEVICE)


In [ ]:
# get the paramaters from the pretrained model, and freeze them
teacher = torch.load('model_032_updated.pt')

leaves = [n for n in dG.nodes() if dG.out_degree(n) == 0]

for p in teacher.parameters():
    p.requires_grad = False



In [ ]:
model_loaded = torch.load('model_032_updated.pt', map_location='cpu')

state_dict = model_loaded.state_dict()
current_state_dict = model.state_dict()

filtered_state_dict = {
    k: v for k, v in state_dict.items()
    if k in current_state_dict and v.size() == current_state_dict[k].size()
}

model.load_state_dict(filtered_state_dict, strict=False)


In [ ]:
def freeze_term_modules(model):
    for name, param in model.named_parameters():
        if (
            any(x in name for x in ['_linear_layer', '_batchnorm_layer'])
            and 'final' not in name
        ):
            param.requires_grad = False

In [ ]:
model.to(DEVICE)
learning_rate = 0.003
torch.manual_seed(0)
loss_list = []
accu_list = []
train_epochs = 500

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.99), eps=1e-05, weight_decay = 1e-4)

term_mask_map = create_term_mask(model.term_direct_gene_map, gene_dim=num_genes, device=DEVICE)

freeze_term_modules(model)

optimizer.zero_grad()

best_epoch = 0
best_accu = 0
best_model_path = "model_classification_freeze_encoder.pt"

for name, param in model.named_parameters():
    term_name = name.split('_')[0]

    if '_direct_gene_layer.weight' in name:
        param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 1
    else:
        param.data = param.data * 1

tepoch = tqdm.tqdm(range(train_epochs))
for epoch in tepoch:

    # Train
    model.train()
    train_predict = torch.zeros(0, 0).to(DEVICE)

    for i, (data, labels) in enumerate(train_loader):
        # Convert torch tensor to Variable

        # Forward + Backward + Optimize
        optimizer.zero_grad()  # zero the gradient buffer

        # Here term_NN_out_map is a dictionary
        logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(data.to(DEVICE))
        
        student_feats = term_NN_out_map

        if train_predict.size()[0] == 0:
            train_predict = aux_out_map["final"].data
        else:
            train_predict = torch.cat([train_predict, aux_out_map["final"].data], dim=0)

        total_loss = 0

        loss_vae, class_loss, KLD = model.loss_log_vae(
            logits=logits, y=labels.to(DEVICE), mu=mu, log_var=log_var, beta=0.001
        )

        loss_intermidiate = model.intermediate_loss_cancer(aux_cancer_map, labels.to(DEVICE))

        total_loss = torch.mean(loss_vae)

        tmp_loss = total_loss.item()
        
        total_loss.backward()

        for name, param in model.named_parameters():
            if "_direct_gene_layer.weight" not in name:
                continue
            term_name = name.split("_")[0]
            # print name, param.grad.data.size(), term_mask_map[term_name].size()
            if param.requires_grad and param.grad is not None:
                term_name = name.split("_")[0]
                param.grad.data = torch.mul(param.grad.data, term_mask_map[term_name])

        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        (inputdata, labels) = next(iter(test_loader))
        inputdata = inputdata.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(inputdata)
        preds = torch.argmax(logits, dim=1)
        accu = (preds == labels).float().mean().item()
    
        loss_list.append(tmp_loss)
        accu_list.append(accu)

    # if epoch % 10 == 0:
    if accu > best_accu:
        best_epoch = epoch
        best_accu = accu
        #torch.save(model, best_model_path)
    tepoch.set_postfix({"Epoch": epoch, "Loss": tmp_loss, "Accuracy": accu})
        
print(f"Epoch {best_epoch}: New best model saved with accuracy {best_accu:.4f}")
print("Training complete. Best model saved at:", best_model_path)


In [ ]:
print(f"Epoch {best_epoch}: New best model saved with accuracy {best_accu:.4f}")
print("Training complete. Best model saved at:", best_model_path)


In [ ]:
with torch.no_grad():
    (inputdata, labels) = next(iter(test_loader))
    inputdata = inputdata.to(DEVICE, non_blocking=True)
    labels = labels.to(DEVICE, non_blocking=True)

    logits, mu, log_var, aux_out_map, aux_cancer_map, term_NN_out_map, term_original_children = model(inputdata)
    preds = torch.argmax(logits, dim=1)
    accu = (preds == labels).float().mean().item()

print(accu)

In [ ]:
with open('tcga_loss_list_freeze_encoder.txt', 'w') as f:
    for loss in loss_list:
        f.write(f"{loss}\n")
    

with open('tcga_accuracy_list_freeze_encoder.txt', 'w') as f:
    for loss in accu_list:
        f.write(f"{loss}\n")

In [ ]:
plt.plot(loss_list)

In [ ]:
plt.plot(accu_list)

In [ ]:
f = model.inter_loss_penalty * 0.1 * loss_intermidiate_cancer
print(torch.mean(f))
print(torch.mean(loss_vae))

In [ ]:
stop

In [ ]:
plt.plot(accu_list)

In [ ]:
plt.plot(accu_list)

In [ ]:
plt.plot(loss_list)

In [ ]:
labels

In [ ]:
total_loss.shape

In [ ]:
loss_vae

In [ ]:
loss_intermidiate

In [ ]:
len(aux_out_map)

In [ ]:
torch.argmax(recon, 1)

In [ ]:
labels

In [ ]:
len(testing_set)

In [ ]:
labels.shape

In [ ]:
torch.argmax(recon, 1).cpu()

## Train an AE for comparison

In [ ]:
torch.manual_seed(0)
train_epochs = 200

train_loader = DataLoader(training_set, batch_size=128, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=1024, shuffle=False)

In [ ]:
model_AE = dcell_vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(cancer_2_idx), turn_off_variational=True)
model_AE.to(DEVICE)
learning_rate = 0.001
torch.manual_seed(0)
loss_list = []
accu_list = []

optimizer = torch.optim.Adam(model_AE.parameters(), lr=learning_rate, betas=(0.9, 0.99), eps=1e-05)
term_mask_map = create_term_mask(model_AE.term_direct_gene_map, gene_dim=num_genes, device=DEVICE)

optimizer.zero_grad()

for name, param in model_AE.named_parameters():
    term_name = name.split('_')[0]

    if '_direct_gene_layer.weight' in name:
        param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 0.1
    else:
        param.data = param.data * 0.1

tepoch = tqdm.tqdm(range(train_epochs))
for epoch in range(train_epochs):
    # Train
    model_AE.train()
    train_predict = torch.zeros(0, 0).to(DEVICE)

    for i, (data, labels) in tqdm.tqdm(enumerate(train_loader)):
        # Convert torch tensor to Variable

        # Forward + Backward + Optimize
        optimizer.zero_grad()  # zero the gradient buffer

        # Here term_NN_out_map is a dictionary
        recon, mu, log_var, aux_out_map, term_NN_out_map = model_AE(data.to(DEVICE))

        if train_predict.size()[0] == 0:
            train_predict = aux_out_map["final"].data
        else:
            train_predict = torch.cat([train_predict, aux_out_map["final"].data], dim=0)

        total_loss = 0

        loss_vae = model_AE.loss_log_vae(
            recon=recon, y=labels.to(DEVICE), mu=mu, log_var=log_var, beta=0.001
        )

        loss_intermidiate = model_AE.intermediate_loss_cancer(aux_out_map, labels.to(DEVICE))

        total_loss = torch.mean(loss_vae + model_AE.inter_loss_penalty * loss_intermidiate)
        
        tmp_loss = total_loss.item()
        
        total_loss.backward()

        for name, param in model_AE.named_parameters():
            if "_direct_gene_layer.weight" not in name:
                continue
            term_name = name.split("_")[0]
            # print name, param.grad.data.size(), term_mask_map[term_name].size()
            param.grad.data = torch.mul(param.grad.data, term_mask_map[term_name])

        optimizer.step()
    
    (inputdata, labels) = next(iter(test_loader))
    recon, mu, log_var, aux_out_map, term_NN_out_map = model_AE(inputdata.to(DEVICE))

    accu = torch.sum(torch.argmax(recon, 1).cpu() == labels)/len(labels)
    
    tepoch.set_postfix({"Epoch": epoch, "Loss": tmp_loss, "Accuracy": accu.item()})
    
    loss_list.append(tmp_loss)
    accu_list.append(accu.item())
    # if epoch % 10 == 0:
    torch.save(model_AE, "model_AE_200.pt")

In [ ]:
i

In [ ]:
import pickle 

with open('loss_accu_AE.pkl', 'wb') as f:
    pickle.dump({'loss': loss_list,
                'accu': accu_list}, f)

### T-SNE/UMAP

In [ ]:
import umap
from sklearn.manifold import TSNE

import seaborn as sns


In [ ]:
test_loader = DataLoader(testing_set, batch_size=len(testing_set), shuffle=False)
(inputdata, labels) = next(iter(test_loader))

In [ ]:
torch.manual_seed(0)
recon, mu, log_var, aux_out_map, term_NN_out_map = model(inputdata.to(DEVICE))

In [ ]:
labels

In [ ]:
torch.sum(torch.argmax(recon, 1).cpu() == labels)/len(labels)

In [ ]:
latent_param = term_NN_out_map['final'].detach().cpu()
mu = term_NN_out_map['final'][..., :model.num_hiddens_final].detach().cpu()
log_var = term_NN_out_map['final'][..., :model.num_hiddens_final].detach().cpu()

In [ ]:
labels_str = [idx_2_cancer[idx.item()] for idx in labels]

In [ ]:
np.random.seed(0)

tsne = TSNE(n_components=2, perplexity=30)

# Fit the model to the data
tsne_tcga_data = tsne.fit_transform(mu.numpy())

In [ ]:
np.random.seed(0)

# Unique category labels: 'D', 'F', 'G', ...
color_labels = np.unique(labels_str)

# List of RGB triplets
rgb_values = sns.color_palette("husl", len(color_labels))

# Map label to RGB
color_map = dict(zip(color_labels, rgb_values))

fig, ax = plt.subplots(1,1, figsize=(3, 2.5))
for category in color_labels:
    mask = np.array(labels_str) == category
    ax.scatter(tsne_tcga_data[mask, 0], tsne_tcga_data[mask, 1], s=10,
            color=color_map[category], label=category)

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(loc='upper center', bbox_to_anchor=(1.7, 1.07), ncol=3)

plt.savefig('tcga_tsne.pdf', bbox_inches='tight')

In [ ]:
np.random.seed(0)

umap_tcga_data = umap.UMAP(n_components=2).fit_transform(mu.numpy())

In [ ]:
np.random.seed(0)

# Unique category labels: 'D', 'F', 'G', ...
color_labels = np.unique(labels_str)

# List of RGB triplets
rgb_values = sns.color_palette("husl", len(color_labels))

# Map label to RGB
color_map = dict(zip(color_labels, rgb_values))

fig, ax = plt.subplots(1,1, figsize=(3, 2.5))
for category in color_labels:
    mask = np.array(labels_str) == category
    ax.scatter(umap_tcga_data[mask, 0], umap_tcga_data[mask, 1], s=10,
            color=color_map[category], label=category)

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(loc='upper center', bbox_to_anchor=(1.7, 1.07), ncol=3)

plt.savefig('tcga_umap.pdf', bbox_inches='tight')

## Model Evaluation

In [ ]:
test_loader = DataLoader(testing_set, batch_size=len(testing_set), shuffle=False)

In [ ]:
model = torch.load("model_200_updated.pt", map_location=DEVICE)
model.eval()

In [ ]:
(inputdata, labels) = next(iter(test_loader))

In [ ]:
recon, mu, log_var, aux_out_map, term_NN_out_map = model(inputdata.to(DEVICE))

In [ ]:
torch.sum(torch.argmax(recon, 1).cpu() == labels)/len(labels)

### Classifiaction accuracy of ontology processes

In [ ]:
term_accu_dict = {}

for name, output in aux_out_map.items():
    if name == 'final':
        continue
    else: # change 0.2 to smaller one for big terms
        ori_y_shape = labels.shape

        term_accu_dict[name] = (torch.sum(torch.argmax(output, 1).cpu() == labels)/len(labels)).item()


In [ ]:
sorted(term_accu_dict, key=term_accu_dict.get)[:5]

In [ ]:
plt.hist(term_accu_dict.values())

### Classification accuracy per cancer type

In [ ]:
labels[labels == 21]

In [ ]:
pred_res = torch.argmax(recon, 1).cpu() == labels

In [ ]:
accu_per_type = {}

for cancer_type in range(33):
    accu_per_type[idx_2_cancer[cancer_type]] = (pred_res[labels == cancer_type]).float().mean().item()

In [ ]:
accu_per_type

In [ ]:
pd.DataFrame([list(accu_per_type.keys()) , list(accu_per_type.values()), torch.bincount(labels).tolist()]).T.sort_values(by=1)

In [ ]:
cancer_2_idx['LUAD']

In [ ]:
(torch.argmax(recon, 1).cpu()[labels == 8])

In [ ]:
idx_2_cancer[19]

### TNSE for latent embedding mean

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
encoder_out = model.encoder(inputdata.to(DEVICE))

# Sankey plot

In [ ]:
import plotly

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15,
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = ["A1", "A2", "B1", "B2", "C1", "C2"],
      color = "blue"
    ),
    link = dict(
      source = [0, 1, 0, 2, 3, 3], # indices correspond to labels, eg A1, A2, A1, B1, ...
      target = [2, 3, 3, 4, 4, 5],
      value = [8, 4, 2, 8, 4, 2]
  ))])

fig.update_layout(title_text="Basic Sankey Diagram", font_size=10)
fig.show()

In [ ]:
dG.nodes()

In [ ]:
model.term_layer_list[-1]

In [ ]:
leaves = [n for n in dG.nodes() if dG.out_degree(n) == 0]

In [ ]:
leaves

In [ ]:
model.term_neighbor_map['GO:0033033']

In [ ]:
model.term_dim_map

In [ ]:
model._modules['GO:0098813_linear_layer']

In [ ]:
dG_tmp = copy.deepcopy(dG)

term_layer_list = []   # term_layer_list stores the built neural network, for nodes of sankey
term_neighbor_map = {}


# term_neighbor_map records all children of each term
for term in dG.nodes():
    term_neighbor_map[term] = []
    for child in dG.neighbors(term):
        term_neighbor_map[term].append(child)

while True:
    leaves = [n for n in dG.nodes() if dG.out_degree(n) == 0]
    #leaves = [n for n,d in dG.out_degree().items() if d==0]
    #leaves = [n for n,d in dG.out_degree() if d==0]

    if len(leaves) == 0:
        break

    self.term_layer_list.append(leaves)

    for term in leaves:

        # input size will be #chilren + #genes directly annotated by the term
        input_size = 0

        for child in self.term_neighbor_map[term]:
            input_size += self.term_dim_map[child]

        if term in self.term_direct_gene_map:
            input_size += len(self.term_direct_gene_map[term])

        # term_hidden is the number of the hidden variables in each state
        term_hidden = self.term_dim_map[term]

        self.add_module(term+'_linear_layer', nn.Linear(input_size, term_hidden))
        self.add_module(term+'_batchnorm_layer', nn.BatchNorm1d(term_hidden))
        self.add_module(term+'_aux_linear_layer1', nn.Linear(term_hidden, self.n_class))
        self.add_module(term+'_aux_linear_layer2', nn.Linear(self.n_class, self.n_class))

    dG.remove_nodes_from(leaves)

In [ ]:
model.term_layer_list[-1]

In [ ]:
model.term_layer_list[-1][-1]

In [ ]:
model._modules['GO:0008150_linear_layer']

In [ ]:
len(model.term_neighbor_map['GO:0008150']) * 6

In [ ]:
model.term_direct_gene_map